# OLS on one saved housing simulation

Run the fixture command in README first. This notebook reads the same saved training rows as the model pipeline, verifies their hash, and checks identical OLS output. It fetches no data and generates no fresh simulation. HC3 standard errors are descriptive and do not address all city/landlord dependence. Synthetic time recovery is a test of the specified generator, not an empirical platform effect. Historical notebook and outputs are preserved in docs/historical/.

In [ ]:
from pathlib import Path
import json
import os
import pandas as pd
from housing_model import digest, ols_checks
run_dir = Path(os.environ.get("HOUSING_RUN", "runs/fixture"))
assert json.loads((run_dir / "status.json").read_text())["state"] == "complete"
provenance = json.loads((run_dir / "provenance.json").read_text())
assert digest(run_dir / "simulation.csv") == provenance["simulation_sha256"]
df = pd.read_csv(run_dir / "simulation.csv")
print({"rows": len(df), "train_rows": int(df.split.eq("train").sum()), "seed": provenance["seed"], "curvature": provenance["curvature"]})

In [ ]:
results = ols_checks(df)
assert results == json.loads((run_dir / "ols.json").read_text())
pd.DataFrame(results)

The observed-rent regression excludes rent-derived dispersion and all generated platform variables. Trimming is performed on training rents only. The synthetic regression includes the known quadratic term, exposure, and rent used by the generator. Compare sensitivity runs across seeds and curvatures separately; do not interpret changes in held-out metrics as causal effects or select the best test run.